# Pre-lab: forecasting the Premier League

People legally bet about 20 billion annualing on the English Premier League, and 1 billion per match on average. This is an astronomical sum of money. Perhaps we can data science it.

The EPL has 20 football teams play a total of 380 matches from August to May. It is Double Round-Robin Format: Every team plays each other team twice (home and away). The outcome of each match contributes to a team's point total:

- Win: 3 points
- Draw: 1 point
- Lose: 0 points
  
At the end of the season, the team with the most points wins the Premier League; if a tie, the winner is the one with the bigger difference between goals scored and goals conceded (if you need further tie-breaking in what follows, decide the winner at random). If you are one of the worst three teams (18th, 19th, 20th), you get kicked out of the Premier League and demoted to EFL Championship.

Your objective for this lab is: Predict which teams win the 2026-2027 Premier League, and which get relegated.


**Goal.** Use the Poisson likelihood to simulate the rest of the season to get title and relegation odds.




In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gammaln

rng = np.random.default_rng(5030)

df = pd.read_parquet("data/premier.parquet")
df.head()

,date,offense,defense,home_away,goals
0,2015-08-08,Bournemouth,Aston Villa,Home,0
1,2015-08-08,Aston Villa,Bournemouth,Away,1
2,2015-08-08,Chelsea,Swansea,Home,2
3,2015-08-08,Swansea,Chelsea,Away,2
4,2015-08-08,Everton,Watford,Home,2


`premier.parquet` has every season from 2015-16 to 2025-26, two rows per match (one row for each team's goals). **In this pre-lab we only use last season, 2025-26** (380 matches, 760 rows). In Thursday's lab you'll use all of the seasons.

In [ ]:
season = df[df["date"] >= "2025-08-01"].reset_index(drop=True)   # 2025-26 only
print(season["date"].min().date(), "to", season["date"].max().date(), "|", len(season) // 2, "matches")

# 1. The Poisson likelihood

Start with the simplest possible model: every team, in every game, scores goals according to the **same** Poisson distribution,
$$
\text{pr}[X = x \mid \lambda] = \frac{e^{-\lambda}\lambda^x}{x!}.
$$



<div class="alert alert-info">

**1.** Compute $\hat\lambda$ for a single Poisson distribution, then plot its PMF next to the observed histogram of goals.

</div>

# 2. Parameterizing the mean

Every team is not the same, so it doesn't really make sense to fit a single Poisson distribution to our data. Let's parameterize $\lambda$ so it depends on the two teams playing in each game:

$$
\text{goals scored by A vs B} \sim \text{Poisson}(\lambda_{AB})
$$
$$
\log \lambda_{AB} = \text{offense}_A + \text{defense}_B
$$

and for the other row of the same game,
$$
\log \lambda_{BA} = \text{offense}_B + \text{defense}_A .
$$

**This is the log link.** We model $\log\lambda$, not $\lambda$, as a linear function of the coefficients: $\log\lambda_i = x_i\cdot\beta$, so $\lambda_i = e^{x_i\cdot\beta}$. In the notes we wrote $\lambda_i = x_i\cdot\beta$ directly (the *identity link*). We use the log link here because:

- **$\lambda$ is always positive.** 
- **It's the standard choice.** It's the default for Poisson regression in `statsmodels` and almost everywhere else.

**Note: For the lab on Thursday, you will also incorporate the `home_away` variable.**

### Design matrix

Each row of $X$ has a 1 in the column of the team on offense, and a 1 in the column of the team on defense. There is **no constant**. We keep all 20 offense columns and **drop one team's defense column** (Arsenal, first alphabetically), just like dropping a category for dummy variables. Otherwise the model isn't identified: adding the same number to every offense coefficient and subtracting it from every defense coefficient gives exactly the same $\lambda$'s, so the MLE isn't unique.

With Arsenal's defense as the baseline ($\text{defense}_{\text{Arsenal}} = 0$), $e^{\text{offense}_A}$ is team A's expected goals against Arsenal's defense, and $e^{\text{defense}_B}$ is how many times more goals team B concedes than Arsenal does.

In [ ]:
def design_matrix(df, teams):
    off = pd.get_dummies(df["offense"]).reindex(columns=teams, fill_value=0).astype(float)
    dfn = pd.get_dummies(df["defense"]).reindex(columns=teams, fill_value=0).astype(float)
    # no constant: keep every offense column, drop the first team's (Arsenal's) defense column
    X = np.column_stack([off, dfn.iloc[:, 1:]])
    names = ["off_" + t for t in teams] + ["def_" + t for t in teams[1:]]
    return X, names

teams = sorted(season["offense"].unique())    # alphabetical, so the baseline is Arsenal
X, names = design_matrix(season, teams)
y = season["goals"].to_numpy()
print(X.shape)   # 760 rows, 20 + 19 = 39 columns

### Log-likelihood and gradient

With $\lambda_i = e^{x_i\cdot\beta}$,
$$
L[y \mid X, \beta] = \sum_{i=1}^n \Big( -e^{x_i\cdot\beta} + y_i\, x_i\cdot\beta - \log(y_i!) \Big)
$$

<div class='alert alert-info'>

**2.** Use the design matrix and statsmodels to find $\lambda$ that maximize the log-likelihood. What does this mean?

</div>

In [ ]:
import statsmodels.api as sm
fit = sm.GLM(y, X, family=sm.families.Poisson()).fit()


# 3. One match = two Poisson distributions

For a match between A and B we get two rates:
$$
\lambda_A = e^{\text{offense}_A + \text{defense}_B}, \qquad
\lambda_B = e^{\text{offense}_B + \text{defense}_A}
$$
If we assume the two teams' goals are independent, the probability of an exact scoreline is
$$
\text{pr}[A = a, B = b] = \text{pr}[A=a]\,\text{pr}[B=b].
$$

<div class='alert alert-info'>

**3.** What is the PMF of number of goals for teams A and B, when team A = "Arsenal" and team B = "Man City"? Which team is predicted to score more goals?

</div>

In [ ]:
# hint: a nice way to get a PMF over two marginal distributions is to use
# grid = np.outer(stats.poisson.pmf(g, lam_a), stats.poisson.pmf(g, lam_b)) 


## Simulate a single match
- Given your fitted model, you can now easily simulate a match. Write a function `simulate_match(home,away,N)` that returns 
 pairs of randomly drawn sets of goals for the home and away teams. (Hint: get the two rates `lam = (lambda_home, lambda_away)` the same way as in **3**, then draw with `rng.poisson(lam, size=(N, 2))`.)

- Write a second function that takes the 
 vector of realized goals, and converts it into a 3-tuple, giving the probability the home team wins, the match is a draw, and the away team wins.

<div class='alert alert-info'>

**4.** Simulate a single match between Arsenal and Man City. 

</div>

## 4. Inference on parameters: bootstrapping

How confident can we be in our estimates? Let's bootstrap and get confidence intervals.

Our $\hat\beta$, and so every $\hat\lambda$, came from one season: 380 games. If those games had gone a little differently, we'd get different ratings and different $\lambda$'s. How different?

Bootstrapping routine:

1. Draw 380 matches **with replacement** from `season`. 
2. Turn the sample into two rows per match and build $X$, exactly as before.
3. Refit the model and compute the two rates for **Arsenal vs Man City**, $\lambda_{\text{Arsenal}}$ and $\lambda_{\text{Man City}}$. Use statsmodels to refit. Calculate the expected goals per team per game.
4. Repeat $B = 1000$ times.
5. Find a 95% confidence interval for the expected goals per team per game, and compare.
6. Plot each bootstrapped distribution.

We resample **matches**, not rows, because a match is what we actually observed, so both teams' goals from one game stay together.

<div class='alert alert-info'>

**5.** Bootstrap $\lambda_{\text{Arsenal}}$ and $\lambda_{\text{Man City}}$ for Arsenal vs Man City using the bootstrapping routine above. Then do the same for Man City vs Wolves.

</div>